In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)

In [2]:
data = {
    'temperature': [10, 15, 22, 25, 28, 35, 38],
    'humidity': [30, 40, 50, 55, 60, 70, 80],
    'y': [0, 0, 1, 1, 1, 0, 0]
}

df = pd.DataFrame(data)
df

,temperature,humidity,y
0,10,30,0
1,15,40,0
2,22,50,1
3,25,55,1
4,28,60,1
5,35,70,0
6,38,80,0


In [3]:
X_raw = df[['temperature', 'humidity']].to_numpy(dtype=np.float32)

X_raw

array([[10., 30.],
       [15., 40.],
       [22., 50.],
       [25., 55.],
       [28., 60.],
       [35., 70.],
       [38., 80.]], dtype=float32)

In [4]:
y = df['y'].to_numpy(dtype=np.float32).reshape(-1, 1)

y

array([[0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.]], dtype=float32)

In [5]:
X_mean = X_raw.mean(axis=0, keepdims=True)
X_std = X_raw.std(axis=0, keepdims=True)

print(f'X_mean={X_mean}, X_std={X_std}')

X_mean=[[24.714285 55.      ]], X_std=[[ 9.345959 15.811388]]


In [6]:
X_norm = (X_raw - X_mean) / X_std

X_norm

array([[-1.5744008 , -1.5811388 ],
       [-1.0394102 , -0.9486833 ],
       [-0.29042336, -0.31622776],
       [ 0.03057098,  0.        ],
       [ 0.35156533,  0.31622776],
       [ 1.1005522 ,  0.9486833 ],
       [ 1.4215466 ,  1.5811388 ]], dtype=float32)

In [7]:
X_tensor = torch.tensor(X_norm, dtype=torch.float32)

X_tensor

tensor([[-1.5744, -1.5811],
        [-1.0394, -0.9487],
        [-0.2904, -0.3162],
        [ 0.0306,  0.0000],
        [ 0.3516,  0.3162],
        [ 1.1006,  0.9487],
        [ 1.4215,  1.5811]])

In [8]:
y_tensor = torch.tensor(y, dtype=torch.float32)

y_tensor

tensor([[0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.]])

In [9]:
# Linear 계층을 만들기 바로 전에 난수를 고정합니다.
# 이렇게 하면 실행할 때마다 Linear 안의 weight, bias 초기값이 똑같이 재현됩니다.
# 즉, 수업 중에 실행 결과가 매번 달라져서 헷갈리는 일을 줄일 수 있습니다.
torch.manual_seed(42)

# hidden_layer:
# 입력 2개(온도, 습도)를 받아 히든 뉴런 2개의 선형값을 만듭니다.
# 직접 구현 방식에서 사람이 직접 만들던 히든층의 weight/bias 역할을
# 이제 torch.nn.Linear가 내부에서 관리합니다.
hidden_layer = torch.nn.Linear(2, 2)

# output_layer:
# 히든 뉴런 2개의 출력을 받아 최종 출력 1개의 선형값을 만듭니다.
# 직접 구현 방식에서 사람이 직접 만들던 출력층의 weight/bias 역할을 
# 이제 torch.nn.Linear가 내부에서 관리합니다.
output_layer = torch.nn.Linear(2, 1)

print('hidden_layer:', hidden_layer)
print('output_layer:', output_layer)

hidden_layer: Linear(in_features=2, out_features=2, bias=True)
output_layer: Linear(in_features=2, out_features=1, bias=True)


In [10]:
def bce_cost_torch(y, z_output):
    epsilon = 1e-7   # 0에 아주 가까운 작은 안전값
    
    # z_output를 [epsilon, 1-epsilon] 범위로 살짝 잘라 log(0)dmf akrtmqslek.
    z_output = torch.clamp(z_output, epsilon, 1 - epsilon)
    
    # 정답이 1일 때(y*log z_output)와 0일 때((1-y)*log(1-z_output))를 더해 평균낸 뒤, -로 뒤집어 Cost로.
    cost = -torch.mean(
        y * torch.log(z_output) + (1 - y) * torch.log(1 - z_output)
    )
    return cost

In [11]:
# 학습 전에 forward를 한 번 실행해서 shape와 cost를 확인합니다. (확인용)
# 아직 학습을 시작하기 전이므로, 여기서 계산되는 Cost는 
# 무작위 초기 weight/bias로 만든 예측 결과의 Cost입니다.

# hidden_layer(X_tensor):
# X_tensor 안에는 입력 2개가 들어 있습니다.
# 여기서 설명을 쉽게 하기 위해 X_tensor의 두 입력 컬럼을 개념적으로 다음처럼 부르겠습니다.
# - X1_norm: 정규화된 온도 입력
# - X2_norm: 정규화된 습도 입력

# 주의:
# X1_norm, X2_norm은 실제 코드에 새로 만드는 변수가 아닙니다.
# X_tensor 안의 첫 번째 입력 컬럼과 두 번째 입력 컬럼을 설명하기 위한 이름입니다.

# 직접 구현 방식에서는 히든 뉴런 2개를 아래처럼 직접 계산했습니다.
# (bias는 곱하는 값이 아니라 마지막에 더하는 값입니다.)

# H_hidden_1 = X1_norm * a11 + X2_norm * a12 + b1
# H_hidden_2 = X1_norm * a21 + X2_norm * a22 + b2

# 이번 Linear/optimizer 방식에서는 이 식을 사람이 직접 코드로 쓰지 않고, 
# hidden_layer가 내부에서 같은 종류의 계산을 한 번에 수행합니다.
H_hidden = hidden_layer(X_tensor)

In [12]:
# 히든층 선형값 H_hidden을 sigmoid에 통과시킵니다.

# 직접 구현 방식의 흐름:
# z_hidden_1 = sigmoid(H_hidden_1)
# z_hidden_2 = sigmoid(H_hidden_2)

# z_hidden은 0~1 사이 값입니다.
# 단, z_hidden은 최종 예측확률이 아닙니다.
# output_layer로 넘어가는 히든층의 중간 출력값입니다.
z_hidden = torch.sigmoid(H_hidden)

In [13]:
# output_layer(z_hidden):
# 히든 뉴런 2개의 출력값을 받아 최종 선형값 H_output을 계산합니다.

# 직접 구현 방식에서는 아래처럼 직접 계산했습니다.

# H_output = z_hidden_1 * a31 + z_hidden_2 * a32 + b3

# 여기서 a31, a32는 출력층 weight 역할이고, 
# b3는 출력층 bias 역할입니다.

# 다만 이번 방식에서는 output_layer가 내부에서 이 계산을 수행합니다.
H_output = output_layer(z_hidden)

In [14]:
# 출력층 선형값 H_output을 sigmoid에 통과시켜 최종 예측확률 z_output을 만듭니다.

# 직접 구현 방식의 흐름:
# z_output = sigmoid(H_output)

# z_output은 최종 예측확률입니다.
# z_output이 0.5 이상이면 클래스 1,
# z_output이 0.5 미만이면 클래스 0으로 예측할 수 있습니다.
z_output = torch.sigmoid(H_output)

In [15]:
# 정답 y_tensor와 예측확률 z_output을 비교해 BCE Cost를 계산합ㅈ니다.

# BCE Cost는 이진 분류에서 예측확률이 정답과 얼마나 다른지 계산하는 값입니다.

# 개념적으로는 다음 흐름입니다.
# y_tensor와 z_output의 차이를 Cost로 계산
# -> Cost가 작아지도록 weight와 bias를 수정

# 이 Cost가 학습을 거치며 줄어드는지 확인할 것입니다.
cost = bce_cost_torch(y_tensor, z_output)

print('H_hidden shape:', H_hidden.shape)
print('z_output shape:', z_output.shape)
print('초기 cost:', cost.item())

H_hidden shape: torch.Size([7, 2])
z_output shape: torch.Size([7, 1])
초기 cost: 0.7983762621879578
